In [1]:
import pandas as pd
# read the CSV files

#df_infer = pd.read_csv('../data/dice_score_on_manualauto_arm_removed.csv')
df_infer = pd.read_csv('../data/testng-dice.csv')

print(df_infer)


    Unnamed: 0             patient_id  C3_slice_expert  C3_slice_auto  \
0            0  mdacc_HNSCC-01-0479_C              139            140   
1            0  mdacc_HNSCC-01-0480_C              149            148   
2            0  mdacc_HNSCC-01-0481_C              158            158   
3            0  mdacc_HNSCC-01-0482_C               50             54   
4            0  mdacc_HNSCC-01-0483_C              139            141   
..         ...                    ...              ...            ...   
88           0  mdacc_HNSCC-01-0573_C               57             59   
89           0  mdacc_HNSCC-01-0574_C              121            122   
90           0  mdacc_HNSCC-01-0576_C               75             74   
91           0  mdacc_HNSCC-01-0578_C               49             48   
92           0  mdacc_HNSCC-01-0579_C              178            179   

    C3_slice_delta  muscle_dice  precsion  recall  Expert_Muscle_Area  \
0               -1         0.91      0.99    0.84 

In [2]:

def precision(gt, pr):
    TP = np.logical_and(gt, pr).sum()
    FP = pr[(pr==1) & (gt==0)].sum()
    deno = TP+FP
    if deno == 0:
        return np.NaN
    return TP/deno


def recall(gt, pr):
    TP = np.logical_and(gt, pr).sum()
    FN = gt[(gt==1) & (pr==0)].sum()
    deno = TP+FN
    if deno == 0:
        return np.NaN
    return TP/deno


In [8]:
from scripts.image_processing.image_window import get_image_path_by_id,apply_window
from scripts.image_processing.slice_array_from_nifty import get_C3_seg_array_by_id
from scripts.image_processing.slice_area_density import get_c3_slice_area,get_c3_slice_density
import SimpleITK as sitk
import numpy as np
import pandas as pd
from pprint import pprint
from scripts.losses import dice_coef_multiclass_2D
from scripts.image_processing.image_window import get_image_path_by_id


df_init = pd.DataFrame()
df_init_icc = pd.DataFrame()


manual_seg_dir = '../data/raw-data/expert_segmentations/'
auto_seg_dir ='../data/test/output_segmentation/'


#Calculate the Dice scores and save the data

for idx in range(df_infer.shape[0]):
# for idx in range(3):
    patient_id =df_infer.iloc[idx,1][:21]
    c3_slice_manual = df_infer.iloc[idx,-8]
    print(c3_slice_manual)
    c3_slice_auto = df_infer.iloc[idx,-7]
    print(c3_slice_auto)  
    
    
    muscle_manual_seg,sfat_manual_seg,vfat_manual_seg = \
                            get_C3_seg_array_by_id(patient_id,c3_slice_manual,manual_seg_dir)
    muscle_auto_seg,sfat_auto_seg,vfat_auto_seg = \
                            get_C3_seg_array_by_id(patient_id,c3_slice_auto,auto_seg_dir)

    muscle_dice = (2*np.sum(muscle_manual_seg*muscle_auto_seg))  \
                            /(np.sum(muscle_manual_seg)+np.sum(muscle_auto_seg))
    pre = precision(muscle_manual_seg,muscle_auto_seg)
    rec = recall(muscle_manual_seg, muscle_auto_seg)
    
    muscle_manual_area,sfat_manual_area,vfat_manual_area = \
                            get_c3_slice_area(patient_id,c3_slice_manual,manual_seg_dir)  

    
    muscle_auto_area,sfat_auto_area,vfat_auto_area = \
                            get_c3_slice_area(patient_id,c3_slice_auto,auto_seg_dir)  

    
    
    # sfat_dice = (2*np.sum(sfat_manual_seg*sfat_auto_seg))  \
   #                         /(np.sum(sfat_manual_seg)+np.sum(sfat_auto_seg))
   # vfat_dice = (2*np.sum(vfat_manual_seg*vfat_auto_seg))  \
   #                         /(np.sum(vfat_manual_seg)+np.sum(vfat_auto_seg))
    
    df_inter = pd.DataFrame({'patient_id':patient_id,
                                'C3_slice_expert':c3_slice_manual,
                                'C3_slice_auto':c3_slice_auto,
                                'C3_slice_delta':(c3_slice_manual - c3_slice_auto),
                                'muscle_dice':round(muscle_dice,2),
    #                             'sfat_dice':round(sfat_dice,4),
    #                             'vfat_dice':round(vfat_dice,4),
                                'precsion':round(pre,2),
                                'recall':round(rec,2),
                                'Expert_Muscle_Area':round(muscle_manual_area,2),
                                'Auto_Muscle_Area':round(muscle_auto_area,2)                                                     
                            },index=[0])
    
    ###Code block to build out dataframe for ICC Calculation
    
    df_infer_icc_m = pd.DataFrame({'patient_id':patient_id,'muscle_plot':'manual', 'muscle_area':round(muscle_manual_area, 2)},index=[0])
    df_init_icc = df_init_icc.append(df_infer_icc_m)
    df_infer_icc_a = pd.DataFrame({'patient_id':patient_id,'muscle_plot':'auto', 'muscle_area':round(muscle_auto_area, 2)},index=[0])
    df_init_icc = df_init_icc.append(df_infer_icc_a)        
                
    
    
    df_init = df_init.append(df_inter)
    df_init.to_csv('../data/testing-dice.csv')
    print(idx+1,'th',patient_id,'dice saved')
    print(muscle_dice)
    if (muscle_dice< 0.8):print('muscle_dice',muscle_dice,patient_id)
    print()

139
140
0.25
19938.0
0.25
17063.0
1 th mdacc_HNSCC-01-0479_C dice saved
0.9104618794086646

149
148


/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_m)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_a)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:87: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init = df_init.append(df_inter)


0.25
15365.0
0.25
13680.0
2 th mdacc_HNSCC-01-0480_C dice saved
0.914787398863832

158
158


/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_m)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_a)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:87: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init = df_init.append(df_inter)


0.25
21072.0
0.25
18890.0
3 th mdacc_HNSCC-01-0481_C dice saved
0.9357889995495721

50
54


...

/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_m)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_a)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:87: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init = df_init.append(df_inter)


0.25
18768.0
0.25
15783.0
92 th mdacc_HNSCC-01-0578_C dice saved
0.8721021099244595

178
179


/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_m)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_a)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:87: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init = df_init.append(df_inter)


0.25
24286.0
0.25
21859.0
93 th mdacc_HNSCC-01-0579_C dice saved
0.9222667677971611



/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_m)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:83: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init_icc = df_init_icc.append(df_infer_icc_a)
/var/folders/8d/xnhxlwmd2613z09z2_x8_prw0000gn/T/ipykernel_6409/843144989.py:87: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_init = df_init.append(df_inter)


In [9]:
import pingouin as pg
print(df_init_icc)
icc = pg.intraclass_corr(data=df_init_icc, targets='patient_id', raters='muscle_plot', ratings='muscle_area' )
icc.set_index('Type')

               patient_id muscle_plot  muscle_area
0   mdacc_HNSCC-01-0479_C      manual        49.84
0   mdacc_HNSCC-01-0479_C        auto        42.66
0   mdacc_HNSCC-01-0480_C      manual        38.41
0   mdacc_HNSCC-01-0480_C        auto        34.20
0   mdacc_HNSCC-01-0481_C      manual        52.68
..                    ...         ...          ...
0   mdacc_HNSCC-01-0576_C        auto        41.09
0   mdacc_HNSCC-01-0578_C      manual        46.92
0   mdacc_HNSCC-01-0578_C        auto        39.46
0   mdacc_HNSCC-01-0579_C      manual        60.72
0   mdacc_HNSCC-01-0579_C        auto        54.65

[186 rows x 3 columns]


,Description,ICC,F,df1,df2,pval,CI95%
Type,,,,,,,
ICC1,Single raters absolute,0.845719,11.963362,92,93,2.343427e-27,"[0.78, 0.89]"
ICC2,Single random raters,0.855956,151.953750,92,92,4.981753e-75,"[-0.02, 0.97]"
ICC3,Single fixed raters,0.986924,151.953750,92,92,4.981753e-75,"[0.98, 0.99]"
ICC1k,Average raters absolute,0.916411,11.963362,92,93,2.343427e-27,"[0.87, 0.94]"
ICC2k,Average random raters,0.922388,151.953750,92,92,4.981753e-75,"[-0.05, 0.98]"
ICC3k,Average fixed raters,0.993419,151.953750,92,92,4.981753e-75,"[0.99, 1.0]"


/Users/yashravipati/miniforge3/envs/env_tf1/lib/python3.9/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package outdated is out of date. Your version is 0.2.1, the latest is 0.2.2.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(
/Users/yashravipati/miniforge3/envs/env_tf1/lib/python3.9/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.2, the latest is 0.5.3.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(
